In [3]:
import pandas as pd
import numpy as np
from evidently import Dataset, DataDefinition
from evidently.presets import DataDriftPreset
from evidently import Report
import joblib
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CARREGAR DADOS PARA ANÁLISE DE DRIFT
# ============================================================
df = pd.read_parquet('../data/gold/dataset_features_v4.parquet')
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').dropna().reset_index(drop=True)

# Referência = primeiros 70% | Atual = últimos 30%
corte = int(len(df) * 0.7)
df_ref = df.iloc[:corte].copy()
df_cur = df.iloc[corte:].copy()

print(f"Dataset carregado: {df.shape}")
print(f"Referência: {len(df_ref)} registros ({df_ref['data'].min().date()} → {df_ref['data'].max().date()})")
print(f"Atual:      {len(df_cur)} registros ({df_cur['data'].min().date()} → {df_cur['data'].max().date()})")

# Features para monitorar (sem target e sem data)
drop_cols = ['data', 'casos', 'casos_nowcast', 'municipio_id', 
             'casos_por_100k', 'casos_nowcast_por_100k', 'fator_nowcasting']
drop_cols = [c for c in drop_cols if c in df.columns]

features = [c for c in df.columns if c not in drop_cols]
print(f"\nFeatures monitoradas: {len(features)}")

Dataset carregado: (2182, 67)
Referência: 1527 registros (2018-04-13 → 2023-03-14)
Atual:      655 registros (2023-03-15 → 2024-12-28)

Features monitoradas: 60


In [2]:
import evidently
print(evidently.__version__)

0.7.21


In [4]:
# ============================================================
# 2. GERAR RELATÓRIO DE DRIFT COM EVIDENTLY
# ============================================================

# Selecionar features principais para monitorar
features_monitorar = [
    'casos', 'temp_media', 'precipitacao_total', 'umidade_media',
    'casos_lag_7d', 'casos_lag_14d', 'casos_lag_28d',
    'ndvi', 'ndwi', 'ndbi_gee', 'oni_index',
    'trends_lag_7d', 'radiacao_mj'
]
features_monitorar = [f for f in features_monitorar if f in df.columns]

df_ref_sel = df_ref[features_monitorar].copy()
df_cur_sel = df_cur[features_monitorar].copy()

# Definir dataset Evidently
data_def = DataDefinition(
    numerical_columns=features_monitorar
)

ref_dataset = Dataset.from_pandas(df_ref_sel, data_definition=data_def)
cur_dataset = Dataset.from_pandas(df_cur_sel, data_definition=data_def)

# Gerar relatório
report = Report(presets=[DataDriftPreset()])
result = report.run(reference_data=ref_dataset, current_data=cur_dataset)

# Salvar HTML
report.save_html('../reports/evidently_drift_report.html')
print("Relatório salvo em reports/evidently_drift_report.html")

# Extrair métricas
result_dict = result.dict()
print(f"\nRelatório gerado com sucesso!")

TypeError: Report.__init__() got an unexpected keyword argument 'presets'

In [5]:
# Verificar API disponível
import evidently
from evidently import Report
help(Report.__init__)

Help on function __init__ in module evidently.core.report:

__init__(self, metrics: List[Union[evidently.core.metric_types.Metric, evidently.core.container.MetricContainer]], metadata: Dict[str, Union[str, Dict[str, str], List[str]]] = None, tags: List[str] = None, model_id: str = None, reference_id: str = None, batch_size: str = None, dataset_id: str = None, include_tests: bool = False)
    Initialize a Report with metrics and optional metadata.
    
    The constructor maps parameters to class attributes. Additional convenience parameters
    (`model_id`, `reference_id`, `batch_size`, `dataset_id`) are stored in the `metadata` dictionary.



In [6]:
# ============================================================
# 2. GERAR RELATÓRIO DE DRIFT — API Evidently 0.7.x
# ============================================================
from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataDriftPreset

data_def = DataDefinition(numerical_columns=features_monitorar)

ref_dataset = Dataset.from_pandas(df_ref_sel, data_definition=data_def)
cur_dataset = Dataset.from_pandas(df_cur_sel, data_definition=data_def)

# API correta 0.7.x — metrics em vez de presets
report = Report(metrics=[DataDriftPreset()])
result = report.run(reference_data=ref_dataset, current_data=cur_dataset)

# Salvar HTML
report.save_html('../reports/evidently_drift_report.html')
print("Relatório salvo em reports/evidently_drift_report.html")

AttributeError: 'Report' object has no attribute 'save_html'

In [8]:
# Salvar relatório corretamente
result.save_html('../reports/evidently_drift_report.html')
print("Relatório salvo em reports/evidently_drift_report.html")

# Extrair métricas resumidas
result_dict = result.dict()
print(f"\nChaves do resultado: {list(result_dict.keys())}")

Relatório salvo em reports/evidently_drift_report.html

Chaves do resultado: ['metrics', 'tests']


In [11]:
# ============================================================
# 3. EXTRAIR MÉTRICAS — estrutura real
# ============================================================
print("="*60)
print("RELATÓRIO DE DRIFT — Evidently v0.7.x")
print("Referência: 2018-2023 | Atual: 2023-2024")
print("="*60)

for m in metricas:
    nome = m.get('metric_name', '')
    valor = m.get('value', {})
    print(f"\n📊 {nome}")
    if isinstance(valor, dict):
        for k, v in valor.items():
            if isinstance(v, float):
                print(f"   {k}: {v:.3f}")
            else:
                print(f"   {k}: {v}")

# Resumo executivo
drift_count = next((m['value'].get('count', 0) 
                    for m in metricas 
                    if 'DriftedColumnsCount' in m.get('metric_name', '')), 0)
drift_share = next((m['value'].get('share', 0) 
                    for m in metricas 
                    if 'DriftedColumnsCount' in m.get('metric_name', '')), 0)

print(f"\n{'='*60}")
print(f"RESUMO EXECUTIVO:")
print(f"  Features com drift: {int(drift_count)}/{len(features_monitorar)}")
print(f"  Share de drift:     {drift_share*100:.0f}%")

if drift_share >= 0.5:
    print(f"  ⚠️  DRIFT SIGNIFICATIVO — retreino recomendado!")
else:
    print(f"  ✅  Modelo estável — sem drift crítico")

# Abrir relatório HTML
import webbrowser, os
caminho = os.path.abspath('../reports/evidently_drift_report.html')
webbrowser.open(f'file:///{caminho}')
print(f"\n✅ Relatório HTML aberto no navegador")

RELATÓRIO DE DRIFT — Evidently v0.7.x
Referência: 2018-2023 | Atual: 2023-2024

📊 DriftedColumnsCount(drift_share=0.5)
   count: 13.000
   share: 1.000

📊 ValueDrift(column=casos,method=Wasserstein distance (normed),threshold=0.1)

📊 ValueDrift(column=temp_media,method=Wasserstein distance (normed),threshold=0.1)

📊 ValueDrift(column=precipitacao_total,method=Wasserstein distance (normed),threshold=0.1)

📊 ValueDrift(column=umidade_media,method=Wasserstein distance (normed),threshold=0.1)

📊 ValueDrift(column=casos_lag_7d,method=Wasserstein distance (normed),threshold=0.1)

📊 ValueDrift(column=casos_lag_14d,method=Wasserstein distance (normed),threshold=0.1)

📊 ValueDrift(column=casos_lag_28d,method=Wasserstein distance (normed),threshold=0.1)

📊 ValueDrift(column=ndvi,method=Wasserstein distance (normed),threshold=0.1)

📊 ValueDrift(column=ndwi,method=Wasserstein distance (normed),threshold=0.1)

📊 ValueDrift(column=ndbi_gee,method=Wasserstein distance (normed),threshold=0.1)

📊 Value

In [10]:
# Inspecionar estrutura real
print(f"Total métricas: {len(metricas)}")
print(f"\nPrimeira métrica:")
print(json.dumps(metricas[0], indent=2, default=str))

Total métricas: 14

Primeira métrica:
{
  "id": "15e89f895b482f9b84ba7274ed18a106",
  "metric_name": "DriftedColumnsCount(drift_share=0.5)",
  "config": {
    "type": "evidently:metric_v2:DriftedColumnsCount",
    "drift_share": 0.5
  },
  "value": {
    "count": 13.0,
    "share": 1.0
  }
}
